# GitHub OSS Founder Departure Dataset Demo

This notebook demonstrates the generation of a synthetic dataset for studying founder departure in open-source software projects.

Based on Avelino et al. (2019) findings, this dataset includes:
- Monthly time series of founder's commit share from inception to departure
- Static snapshot features at departure (bus factor, contributor count, project age, star count, file count)
- Binary survival label (survived/collapsed) based on sustained non-founder activity post-departure
- Continuous survival metric (post/pre-departure commit ratio)
- Metadata for diversity (domain, governance model, primary language)

The dataset is designed for hypothesis testing on what determines whether OSS projects survive founder departure.

In [ ]:
# Install dependencies - following aii-colab-aii-colab pattern
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install everywhere)
_pip('loguru==0.7.2')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2')


In [ ]:
# Imports - copy original import block as-is
from loguru import logger
from pathlib import Path
import json
import sys
import tarfile
import gzip
import csv
import io

# Additional imports for notebook visualization
import matplotlib.pyplot as plt
import numpy as np

# NumPy 2.0 compatibility shim
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

In [ ]:
# Data loading helper - GitHub URL with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-ad55a2-founder-fade-curve-predicts-oss-survival/main/round-2/dataset-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
# Load the data
data = load_data()
print(f"Loaded dataset with {len(data['datasets'][0]['examples'])} examples")

## Configuration

Define tunable parameters with ABSOLUTE MINIMUM values for demo purposes.

In [ ]:
# Configurable parameters - SET TO MINIMUM VALUES FOR DEMO
NUM_PROJECTS = 3  # Reduced from 100
MIN_PROJECT_AGE = 6   # Reduced from 12
MAX_PROJECT_AGE = 12  # Reduced from 180
MIN_INITIAL_CONTRIBUTORS = 1  # Same as original
MAX_INITIAL_CONTRIBUTORS = 3  # Reduced from 25
MIN_TOTAL_COMMITS = 10    # Reduced from 100
MAX_TOTAL_COMMITS = 50    # Reduced from 10000
MIN_STARS = 5             # Reduced from 10
MAX_STARS = 50            # Reduced from 5000
EARLY_DEPARTURE_PROB = 0.59  # Same as original
SURVIVAL_RATE = 0.41      # Same as original

print("Configuration:")
print(f"  Number of projects: {NUM_PROJECTS}")
print(f"  Project age range: {MIN_PROJECT_AGE}-{MAX_PROJECT_AGE} months")
print(f"  Initial contributors: {MIN_INITIAL_CONTRIBUTORS}-{MAX_INITIAL_CONTRIBUTORS}")
print(f"  Total commits: {MIN_TOTAL_COMMITS}-{MAX_TOTAL_COMMITS}")
print(f"  Stars: {MIN_STARS}-{MAX_STARS}")
print(f"  Early departure probability: {EARLY_DEPARTURE_PROB}")
print(f"  Survival rate: {SURVIVAL_RATE}")

## Dataset Generation

Generate synthetic dataset based on research findings (adapted from original data.py with minimal values).

In [ ]:
@logger.catch(reraise=True)
def generate_synthetic_dataset():
    """Generate a synthetic dataset based on research findings for demonstration.

    This creates realistic project data based on the Avelino et al. (2019) findings:
    - 16% of projects experience TFDD (Truck Factor Developer Detachment)
    - 41% of abandoned projects survive with new core developers
    - Survival associated with younger projects at TFDD time
    """
    import random
    
    logger.info("Generating synthetic dataset based on research findings...")
    
    # Generate projects with realistic distributions
    projects = []
    
    for i in range(NUM_PROJECTS):
        # Project characteristics based on research
        project_age_months = random.randint(MIN_PROJECT_AGE, MAX_PROJECT_AGE)  # Configurable range
        initial_contributors = random.randint(MIN_INITIAL_CONTRIBUTORS, MAX_INITIAL_CONTRIBUTORS)  # Configurable range
        total_commits = random.randint(MIN_TOTAL_COMMITS, MAX_TOTAL_COMMITS)  # Configurable range
        stars = random.randint(MIN_STARS, MAX_STARS)  # Configurable range
        
        # Founder departure timing (59% within first 2 years per Avelino)
        if random.random() < EARLY_DEPARTURE_PROB:  # Configurable
            founder_departure_month = random.randint(6, min(24, max(6, project_age_months)))
        else:
            if project_age_months >= 25:
                founder_departure_month = random.randint(25, project_age_months)
            else:
                founder_departure_month = random.randint(6, project_age_months)
        
        # Calculate founder's commit share trajectory
        founder_peak_share = random.uniform(0.4, 0.9)
        
        # Monthly founder share with decay pattern
        monthly_founder_shares = []
        for month in range(1, project_age_months + 1):
            if month <= founder_departure_month:
                # Decay pattern: starts high, decreases toward departure
                decay_factor = 1.0 - (month / (founder_departure_month * 1.5))
                share = max(0.1, founder_peak_share * decay_factor + random.gauss(0, 0.1))
            else:
                # After departure: founder has 0% share
                share = 0.0
            monthly_founder_shares.append(round(share, 3))
        
        # Determine survival outcome
        # 41% survival rate for abandoned projects
        if random.random() < SURVIVAL_RATE:  # Configurable
            survival_label = "survived"
            # Post-departure activity continues
            post_departure_commits = random.randint(50, total_commits // 2)
            new_core_contributors = random.randint(1, 5)
        else:
            survival_label = "collapsed"
            post_departure_commits = random.randint(0, 20)  # Minimal activity
            new_core_contributors = 0
        
        # Continuous survival metric (post/pre departure ratio)
        pre_departure_commits = total_commits - post_departure_commits
        if pre_departure_commits > 0:
            survival_metric = round(post_departure_commits / pre_departure_commits, 3)
        else:
            survival_metric = 0.0
        
        # Static features at departure time
        bus_factor_at_departure = max(1, int(initial_contributors * random.uniform(0.3, 0.8)))
        contributor_count_at_departure = initial_contributors + random.randint(0, 10)
        
        # Project metadata
        domains = ["web", "systems", "data", "ml", "devtools", "cloud", "security", "cli"]
        domain = random.choice(domains)
        
        governance_models = ["BDFL", "meritocratic", "corporate-backed", "community"]
        governance_model = random.choice(governance_models)
        
        languages = ["Python", "JavaScript", "Go", "Rust", "Java"]
        primary_language = random.choice(languages)
        
        project = {
            "project_id": f"oss_project_{i:03d}",
            "project_name": f"example-project-{i:03d}",
            "founder_username": f"user_{i:04d}",
            
            # Temporal data
            "project_start_date": f"2015-{random.randint(1,12):02d}-01",
            "founder_departure_month": founder_departure_month,
            "founder_departure_date": f"201{founder_departure_month // 12}-{(founder_departure_month % 12) + 1:02d}-01",
            "project_age_months": project_age_months,
            
            # Founder trajectory
            "monthly_founder_commit_share": monthly_founder_shares,
            "founder_peak_share": founder_peak_share,
            "founder_departure_type": random.choice(["gradual", "sudden", "planned"]),
            
            # Survival labels
            "survival_label": survival_label,
            "survival_metric": survival_metric,
            "post_departure_commits": post_departure_commits,
            "pre_departure_commits": pre_departure_commits,
            "new_core_contributors": new_core_contributors,
            
            # Static features at departure
            "bus_factor_at_departure": bus_factor_at_departure,
            "contributor_count_at_departure": contributor_count_at_departure,
            "star_count": stars,
            "file_count": random.randint(10, 50),  # Reduced range
            "total_commits": total_commits,
            
            # Metadata
            "domain": domain,
            "governance_model": governance_model,
            "primary_language": primary_language,
            "hosting_platform": "GitHub",
            
            # Research metadata
            "data_source": "synthetic_based_on_avelino_2019",
            "notes": "Dataset generated based on Avelino et al. (2019) findings and literature review"
        }
        
        projects.append(project)
    }
    
    return projects

@logger.catch(reraise=True)
def transform_to_exp_format(projects: list) -> dict:
    """Transform projects to exp_sel_data_out format."""
    datasets = []
    
    examples = []
    for i, project in enumerate(projects):
        # Create input features
        input_features = {
            "founder_peak_share": project["founder_peak_share"],
            "bus_factor_at_departure": project["bus_factor_at_departure"],
            "contributor_count_at_departure": project["contributor_count_at_departure"],
            "project_age_months": project["project_age_months"],
            "star_count": project["star_count"],
            "file_count": project["file_count"],
            "total_commits": project["total_commits"],
            "governance_model": project["governance_model"],
            "domain": project["domain"],
            "primary_language": project["primary_language"],
        }
        
        # Create output (survival prediction)
        output = {
            "survival_label": project["survival_label"],
            "survival_metric": project["survival_metric"]
        }
        
        example = {
            "input": json.dumps(input_features),
            "output": json.dumps(output),
            "metadata_task_type": "binary_classification",
            "metadata_n_classes": 2,
            "metadata_row_index": i,
            "metadata_feature_names": list(input_features.keys()),
            "metadata_project_id": project["project_id"],
            "metadata_founder_departure_month": project["founder_departure_month"],
            "metadata_post_departure_commits": project["post_departure_commits"],
        }
        
        examples.append(example)
    }
    
    datasets.append({
        "dataset": "oss_founder_departure",
        "examples": examples
    })
    
    return {"datasets": datasets}

# Generate the dataset
logger.info("=" * 60)
logger.info("Starting OSS Founder Departure Dataset Collection")
logger.info("=" * 60)

# Step 1: Generate dataset
projects = generate_synthetic_dataset()

logger.info(f"Generated {len(projects)} projects")

# Step 2: Transform to output format
output = transform_to_exp_format(projects)

# Step 3: Save full data
WORKSPACE = Path(".")
full_path = WORKSPACE / "full_data_out.json"
full_path.write_text(json.dumps(output, indent=2))
logger.info(f"Saved full dataset to {full_path}")

# Step 4: Generate preview (first 2 rows)
preview = {k: v for k, v in output.items()}
if "datasets" in preview:
    for ds in preview["datasets"]:
        ds["examples"] = ds["examples"][:2]
preview_path = WORKSPACE / "preview_data_out.json"
preview_path.write_text(json.dumps(preview, indent=2))
logger.info(f"Saved preview to {preview_path}")

# Step 5: Generate mini (first 10 rows, but we have fewer)
mini = {k: v for k, v in output.items()}
if "datasets" in mini:
    for ds in mini["datasets"]:
        ds["examples"] = ds["examples"][:min(10, len(ds["examples"]))]
mini_path = WORKSPACE / "mini_data_out.json"
mini_path.write_text(json.dumps(mini, indent=2))
logger.info(f"Saved mini dataset to {mini_path}")

logger.info("=" * 60)
logger.info("Dataset collection complete!")
logger.info("=" * 60)

output

## Results and Visualization

Display key results from the generated dataset.

In [ ]:
# Print summary statistics
examples = output['datasets'][0]['examples']
print(f"Generated {len(examples)} project examples\n")

# Count survival outcomes
survived_count = sum(1 for ex in examples if json.loads(ex['output'])['survival_label'] == 'survived')
collapsed_count = len(examples) - survived_count
print(f"Survival outcomes:")
print(f"  Survived: {survived_count} ({survived_count/len(examples)*100:.1f}%)")
print(f"  Collapsed: {collapsed_count} ({collapsed_count/len(examples)*100:.1f}%)\n")

# Show average metrics
avg_founders_peak = np.mean([json.loads(ex['input'])['founder_peak_share'] for ex in examples])
avg_survival_metric = np.mean([json.loads(ex['output'])['survival_metric'] for ex in examples])
avg_project_age = np.mean([json.loads(ex['input'])['project_age_months'] for ex in examples])
print(f"Average metrics:")
print(f"  Founder peak share: {avg_founders_peak:.3f}")
print(f"  Survival metric: {avg_survival_metric:.3f}")
print(f"  Project age: {avg_project_age:.1f} months\n")

# Display first example in detail
first_example = examples[0]
input_data = json.loads(first_example['input'])
output_data = json.loads(first_example['output'])

print("First project example:")
print(f"  Project ID: {first_example['metadata_project_id']}")
print(f"  Founder peak share: {input_data['founder_peak_share']:.3f}")
print(f"  Bus factor at departure: {input_data['bus_factor_at_departure']}")
print(f"  Contributors at departure: {input_data['contributor_count_at_departure']}")
print(f"  Project age: {input_data['project_age_months']} months")
print(f"  Stars: {input_data['star_count']}")
print(f"  Files: {input_data['file_count']}")
print(f"  Total commits: {input_data['total_commits']}")
print(f"  Governance: {input_data['governance_model']}")
print(f"  Domain: {input_data['domain']}")
print(f"  Language: {input_data['primary_language']}")
print(f"  Survival label: {output_data['survival_label']}")
print(f"  Survival metric: {output_data['survival_metric']}")
print(f"  Post-departure commits: {first_example['metadata_post_departure_commits']}")


In [ ]:
# Visualization: Founder share trajectories
examples = output['datasets'][0]['examples']

plt.figure(figsize=(12, 8))

for i, example in enumerate(examples[:3]):  # Plot first 3 examples
    input_data = json.loads(example['input'])
    output_data = json.loads(example['output'])
    founder_shares = json.loads(example['input']).get('monthly_founder_commit_share', [])

    if founder_shares:
        months = list(range(1, len(founder_shares) + 1))
        plt.plot(months, founder_shares, marker='o', linewidth=2, markersize=4, 
                 label=f"{example['metadata_project_id']} ({output_data['survival_label']})")
        # Mark departure point
        departure_month = example['metadata_founder_departure_month']
        if departure_month <= len(founder_shares):
            plt.axvline(x=departure_month, color='red', linestyle='--', alpha=0.7)
            plt.scatter([departure_month], [founder_shares[departure_month-1]], 
                     color='red', s=100, zorder=5)

plt.xlabel('Months since project start')
plt.ylabel('Founder commit share')
plt.title('Founder Commit Share Trajectories (First 3 Projects)\nDashed line indicates founder departure')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Visualization: Survival outcomes
examples = output['datasets'][0]['examples']
survival_labels = [json.loads(ex['output'])['survival_label'] for ex in examples]
survival_metrics = [json.loads(ex['output'])['survival_metric'] for ex in examples]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Survival label distribution
from collections import Counter
label_counts = Counter(survival_labels)
ax1.pie(label_counts.values(), labels=label_counts.keys(), autopct='%1.1f%%', startangle=90)
ax1.set_title('Survival Label Distribution')

# Survival metric distribution
ax2.hist(survival_metrics, bins=10, edgecolor='black', alpha=0.7)
ax2.set_xlabel('Survival Metric (Post/Pre Departure Commit Ratio)')
ax2.set_ylabel('Frequency')
ax2.set_title('Distribution of Survival Metrics')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
